# Add a water year

Notebook to extend the dataset once a new water year becomes processable — driven by the
Water Year Watch issue ([`water_year_watch.yml`](../.github/workflows/water_year_watch.yml)
opens one per hemisphere-year after the season + `trailing_buffer_days` has elapsed). Do two
things before running: check that MODIS_snow_phenology has committed the new year for the
triggering hemisphere (that repo opens its own reminder issue ~1 month before ours), and bump
`WY_end` in the config — the target year, search dates, and `water_years` all derive from it.

The store append itself (step 3, via `store.extend_water_years`) is cheap and safe: shards
are (1 water_year, 2048, 2048), so the resize is shard-aligned and metadata-only — no
existing chunk is touched, and new slots read as fill until tiles write them. The commit
carries no status metadata, so status derivation ignores it: every (tile, new year) simply
shows up as `missing` work for the eligible hemisphere, and the fleet fills it in. Once the
fleet finishes, re-run [`4_finalize_icechunk_store.ipynb`](4_finalize_icechunk_store.ipynb)
with a bumped tag (`v10.1`, ...).

In [ ]:
import pandas as pd
import rioxarray  # noqa: F401 -- .rio accessor for the phenology spot-check
import xarray as xr

from global_snowmelt_runoff_onset.config import Config
from global_snowmelt_runoff_onset import status, store

config = Config('config/global_config_v10.txt')
repo = config.open_output_repo()

target_wy = int(config.WY_end)  # bump WY_end in the config first; everything follows it
print(f'config water years: {config.water_years[0]}..{config.water_years[-1]} '
      f'-> target WY{target_wy}')

## 1. Preconditions

The target year must be hemisphere-eligible (season end + `trailing_buffer_days`), and the
phenology store must actually contain it for that hemisphere. The second check is **the
hemisphere trap**: an all-fill phenology slab is indistinguishable from verified no-snow, so
processing a year whose phenology hasn't been computed yet would durably commit
`no_seasonal_snow` everywhere. `status.wy_eligible` guards this at dispatch time by date;
the spot-check below looks at the phenology *data* itself, before we touch anything.

In [ ]:
eligible = {h: status.wy_eligible(target_wy, h,
                                  trailing_buffer_days=config.trailing_buffer_days)
            for h in ('northern', 'southern')}
for hemisphere, ok in eligible.items():
    print(f'{hemisphere}: season ended {status.season_end(target_wy, hemisphere)}, '
          f'eligible={ok}')
assert any(eligible.values()), f'WY{target_wy} is not yet eligible for either hemisphere'

In [ ]:
# known seasonal-snow windows (lon/lat bounds), one per hemisphere
SNOWY_WINDOWS = {'northern': (5.5, 45.5, 11.0, 47.5),      # European Alps
                 'southern': (-71.5, -35.0, -69.5, -32.0)}  # Chilean Andes

phenology_ds = xr.open_zarr(config.snow_phenology_store, zarr_format=3,
                            consolidated=False, decode_coords='all')
phenology_years = [int(wy) for wy in phenology_ds.water_year.values]
assert target_wy in phenology_years, (
    f'phenology store spans {phenology_years[0]}..{phenology_years[-1]} -- '
    'extend MODIS_snow_phenology first')

min_days = config.min_consec_snow_days_for_seasonal_snow
for hemisphere, ok in eligible.items():
    if not ok:
        print(f'{hemisphere}: not yet eligible, skipping spot-check')
        continue
    window = (phenology_ds['max_consec_snow_days']
              .sel(water_year=target_wy)
              .rio.clip_box(*SNOWY_WINDOWS[hemisphere], crs='EPSG:4326'))
    n_seasonal = int((window >= min_days).sum())
    print(f'{hemisphere} spot-check: {n_seasonal:,} px with >= {min_days} '
          'consecutive snow days')
    assert n_seasonal > 0, (
        f'phenology WY{target_wy} looks all-fill for the {hemisphere} hemisphere -- '
        'processing now would durably commit no_seasonal_snow (the hemisphere trap)')

## 2. Dry run

`store.extend_water_years` discovers the water_year-dimensioned arrays by their zarr
`dimension_names` (the 2-D composites are untouched) and reports what an append would do —
nothing is written.

In [ ]:
plan = store.extend_water_years(config, repo, dry_run=True)
print(f"store water_year : {plan['current_years'][0]}..{plan['current_years'][-1]} "
      f"({len(plan['current_years'])} slots)")
print(f"arrays to resize : {plan['arrays']}")
print(f"years to append  : {plan['new_years'] or 'none -- already extends through target'}")

## 3. Append and commit

Shard-aligned, metadata-only resize of every water_year-dimensioned array plus the
coordinate, in one commit. The function verifies before returning: the coordinate must read
back as expected and a sample of the new slab must be raw fill.

In [ ]:
result = store.extend_water_years(config, repo)
if result['snapshot_id'] is None:
    print('nothing to do -- store already extends through the target year')
else:
    appended = ', '.join(f'WY{wy}' for wy in result['new_years'])
    print(f"appended {appended} -> commit {result['snapshot_id']}")

## 4. Verify the work list

The new (tile, year) items must now appear in the dispatcher's view — for the eligible
hemisphere only. `get_remaining_work` derives everything from config × commit history, so
there is no other bookkeeping to update.

In [ ]:
work = status.get_remaining_work(config, repo=repo)
new_year_items = [item for item in work if target_wy in item['water_years']]
print(f'{len(work):,} tiles with work, {len(new_year_items):,} include WY{target_wy}')

tiles_gdf = status.get_tile_status_gdf(config, repo=repo)
print(tiles_gdf.groupby('hemisphere')[f'wy_{target_wy}'].value_counts().to_string())

## 5. Dispatch the fleet

Nothing year-specific to configure: `get_remaining_work` emits only the missing eligible
years, so the standard fleet entrypoint processes exactly the new year plus the composite
refreshes that the staleness rule triggers. Monitor with
[`2_check_tile_status.ipynb`](2_check_tile_status.ipynb); once everything is complete,
re-run [`4_finalize_icechunk_store.ipynb`](4_finalize_icechunk_store.ipynb) with a bumped
tag.

In [ ]:
print('gh workflow run process_all_tiles.yml -f which_tiles=incomplete')